### Setup

In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 42

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
feature_engineer = FeatureEngineer()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42
Seed set to 42


random seed set to 42
numpy seed set to 42
torch seed set to 42
lightning seed set to 42
torch set to use deterministic algorithms


### Load and Process DataFrame

In [2]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)
interaction_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1


### Join Side Information

In [3]:
interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5,
)
interaction_info_df.head()

extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


### Prepare Train/Valid/Test Set

In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.75,
    val_ratio=0.1,
    test_ratio=0.15,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.75 : 0.1 : 0.15):
train: 358027 (74.84%)
valid: 46916 (9.81%)
test: 73461 (15.36%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.555944
1    0.444056
Name: proportion, dtype: float64
valid label
0    0.610772
1    0.389228
Name: proportion, dtype: float64
test label
0    0.585277
1    0.414723
Name: proportion, dtype: float64


### Re-index User/Item ID & Encode Categorical Features

In [5]:
print("Train: fit_transform")
encoded_train_df = feature_engineer.fit_transform(train_df)
print("---"*10)
print("Valid: transform")
encoded_valid_df = feature_engineer.transform(valid_df)
print("---"*10)
print("Test: transform")
encoded_test_df = feature_engineer.transform(test_df)
print("---"*10)

Train: fit_transform
Re-index mapping dumped into ...
user: ../datasets/userid_mapping.csv
item: ../datasets/itemid_mapping.csv
Fitted: user/item mapping
Fitted: vocab2idx for actorID
Fitted: vocab2idx for country
Fitted: vocab2idx for directorID
Fitted: vocab2idx for genre
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Valid: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------
Test: transform
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
------------------------------


In [6]:
# # NOTE: can check the encoding vocab idx content from the feature engineer
# oov_idx = feature_engineer.vocab2idx["movieID"]["[OOV]"]
# len(test_df[test_df["movieID"] == oov_idx])

### Prepare Additional Data for Train/Inference

#### Build bi-partite graph for training

In [7]:
# NOTE: At training, we use interaction graph from train_df for train and validation
train_graph = experiment_data_preprocessor.create_interaction_graph(encoded_train_df)

# NOTE: At inference, we can use graph of (train_df + valid_df)
# train_valid_graph = utils.create_interaction_graph(pd.concat([train_df, valid_df], axis=0))

Creating interaction graph...
Drop negative samples
  Num of all interactions: 358027
  Num of positive interactions: 158984 

Building edges...
Building labels...
Interaction Graph: Data(edge_index=[2, 317967], edge_label=[158984])
Edge Index: tensor([[    0,     0,     0,  ..., 10758, 10762, 10763],
        [ 2089,  2202,  2318,  ...,  1760,  1645,  1760]])


#### Prepare train/valid triplet data

In [8]:
train_triplet_df = experiment_data_preprocessor.prepare_triplet_df(encoded_train_df, k_negative_samples=10)
train_triplet_df.head(1)

Original data count (positive samples): 158984
Num of triplets: 158984(pos samples) * 10(negative sampled items) = 1589840


,userID,pos_item_id,neg_item_id,actorID_idx,country_idx,directorID_idx,genre_idx,neg_actorID_idx,neg_country_idx,neg_directorID_idx,neg_genre_idx
0,0,1102,307,"[2267, 1401, 582, 998, 1847]",37,360,"[2, 3, 17, 18, 0, 0, 0, 0]","[1769, 713, 931, 1356, 1]",12,452,"[9, 11, 0, 0, 0, 0, 0, 0]"


#### Prepare prediction pool for inference/testing

In [9]:
# NOTE: Prepare prediction pool to evaluate the model
valid_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_valid_df, K=500)
prediction_pool_df = experiment_data_preprocessor.prepare_prediction_df(encoded_test_df, K=500)
prediction_pool_df.tail()

Prediction DataFrame:
User Pool: 2063
Item Pool: 6098, negative sampled to 500 items for each user
Num of interactions: 2063(users) * 500(items) = 1031500
Prediction DataFrame:
User Pool: 2064
Item Pool: 6959, negative sampled to 500 items for each user
Num of interactions: 2064(users) * 500(items) = 1032000


,userID,movieID,label,actorID_idx,country_idx,directorID_idx,genre_idx
1031995,2063,447,0,"[2136, 281, 1446, 61, 1]",36,1,"[9, 0, 0, 0, 0, 0, 0, 0]"
1031996,2063,3601,0,"[1578, 466, 911, 941, 887]",37,432,"[2, 19, 0, 0, 0, 0, 0, 0]"
1031997,2063,7982,0,"[1, 1, 1, 1, 1]",37,1,"[12, 0, 0, 0, 0, 0, 0, 0]"
1031998,2063,5969,0,"[1, 1383, 554, 1, 828]",37,534,"[9, 16, 0, 0, 0, 0, 0, 0]"
1031999,2063,5025,0,"[446, 1010, 637, 809, 547]",36,70,"[7, 12, 15, 18, 0, 0, 0, 0]"


### Prepare DataLoader

In [10]:
# NOTE: ensure reproducibility of DataLoader
import torch
from common.utils import seed_worker
g = torch.Generator()
g.manual_seed(RANDOM_SEED)

# TODO: determine which Dataset to use
from torch.utils.data import DataLoader
from common.datasets import TripletDataset, UserItemPairDataset

BATCH_SIZE = 1024

train_dataset = TripletDataset(train_triplet_df)
valid_dataset = UserItemPairDataset(valid_pool_df)
test_dataset = UserItemPairDataset(prediction_pool_df)
print("train data count:", len(train_dataset))
print("valid data count:", len(valid_dataset))
print("test data count:", len(test_dataset))

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, worker_init_fn=seed_worker, generator=g, num_workers=4)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)


train data count: 1589840
valid data count: 1031500
test data count: 1032000


### Configure Model (LightningModule)

In [11]:
from lightning_models.ngcf import NGCFRec

EMB_DIM = 16
LR = 1e-3
EPOCHS = 1
NUM_LAYERS = 3
REG_WEIGHT = 1e-5

model = NGCFRec(
    graph_data=train_graph,  # shape [2, num_edges]
    num_users=TRAIN_NUM_USERS,
    num_items=TRAIN_NUM_ITEMS,
    embedding_dim=EMB_DIM,
    num_layers=NUM_LAYERS,
    node_dropout=0.0,
    mess_dropout=0.1,
    lr=LR,
    reg_weight=REG_WEIGHT,
)


Seed set to 42


### Configure Trainer and Experiment

In [12]:
from common._mlflow import get_mlflow_logger, get_callbacks

EXPERIMENT_NAME = "ngcf-exp"
RUN_NAME = "test-run"
PATIENCE = 5
VERSION = ""
mlflow_logger = get_mlflow_logger(experiment_name=EXPERIMENT_NAME, run_name=RUN_NAME, tags={"version": VERSION})
trainer_callbacks = get_callbacks(
    exp_name=EXPERIMENT_NAME,
    run_name=RUN_NAME,
    patience=PATIENCE,
    monitor_metric="val_ndcg20",
    monitor_mode="max",
    hyper_param_str=f"n_user={TRAIN_NUM_USERS}-n_item={TRAIN_NUM_ITEMS}-emb_dim={EMB_DIM}-num_layers={NUM_LAYERS}-lr={LR}-reg_weight={REG_WEIGHT}",
)

In [13]:
from pytorch_lightning import Trainer

trainer = Trainer(
    max_epochs=EPOCHS,
    logger=mlflow_logger,
    log_every_n_steps=50,
    callbacks=trainer_callbacks,
    accelerator='auto',  # or 'auto', 'gpu'
    # devices=[0], # if gpu is available
)


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


### Train Model

In [14]:
# # Start training
# trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=valid_loader)


### Inference

In [15]:
# NOTE: the inference model MUST be the same as the training model
best_model_experiment_name = "ngcf-exp"
best_model_checkpoint_path = "test-run-n_user=2064-n_item=8706-emb_dim=16-num_layers=3-lr=0.001-reg_weight=1e-05-best-checkpoint-epoch=00-val_ndcg20=0.34.ckpt"
best_model_path = f"test_checkpoints/{best_model_experiment_name}/{best_model_checkpoint_path}"

model = NGCFRec.load_from_checkpoint(checkpoint_path=best_model_path)
# start inference
trainer.test(model=model, dataloaders=test_loader)


Seed set to 42
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│        test_ndcg10        │    0.40788698196411133    │
│        test_ndcg20        │    0.4317229688167572     │
│        test_ndcg5         │    0.3645264208316803     │
│     test_precision10      │    0.1550387591123581     │
│     test_precision20      │    0.13185562193393707    │
│      test_precision5      │    0.17122092843055725    │
│       test_recall10       │    0.13685140013694763    │
│       test_recall20       │    0.22322721779346466    │
│       test_recall5        │    0.08008497208356857    │
└───────────────────────────┴───────────────────────────┘

🏃 View run test-run at: http://140.112.106.216:3683/#/experiments/6/runs/a09249e1a1a4402da8318372e3fa8635
🧪 View experiment at: http://140.112.106.216:3683/#/experiments/6


[{'test_ndcg5': 0.3645264208316803,
  'test_ndcg10': 0.40788698196411133,
  'test_ndcg20': 0.4317229688167572,
  'test_precision5': 0.17122092843055725,
  'test_precision10': 0.1550387591123581,
  'test_precision20': 0.13185562193393707,
  'test_recall5': 0.08008497208356857,
  'test_recall10': 0.13685140013694763,
  'test_recall20': 0.22322721779346466}]

In [16]:
model.test_results["eval_score_df"].describe()

,user,ndcg@5,recall@5,precision@5,ndcg@10,recall@10,precision@10,ndcg@20,recall@20,precision@20
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.364526,0.080085,0.171221,0.407887,0.136851,0.155039,0.431723,0.223227,0.131856
std,595.969798,0.374585,0.125502,0.197820,0.328277,0.163658,0.154756,0.284842,0.208613,0.119567
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,515.750000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.255958,0.062500,0.050000
50%,1031.500000,0.386853,0.031250,0.200000,0.430677,0.090909,0.100000,0.439295,0.181818,0.100000
75%,1547.250000,0.630930,0.112179,0.200000,0.630930,0.200000,0.200000,0.630930,0.333333,0.200000
max,2063.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.900000,1.000000,1.000000,0.750000


In [ ]:
# To get the reversed encoded eval df
eval_df = evaluator.prepare_evaluation_data(model.test_results, feature_engineer.idx2vocab)
eval_df.head(1)


,user,rec_items,gt_items
0,75,"[4993, 5418, 318, 2959, 45722, 4306, 356, 6934...","[45722, 1233, 110, 2959, 2571]"
1,78,"[4306, 46578, 50872, 8917, 1333, 3507, 628, 31...","[4119, 6993, 8400, 50872]"
2,127,"[8607, 6467, 7123, 7386, 33154, 33834, 41863, ...","[45726, 6958]"
3,170,"[296, 260, 7090, 1291, 2671, 1200, 44555, 1265...","[1222, 3949, 4011, 2542, 8874, 44191, 45728, 3..."
4,175,"[5902, 8917, 380, 7387, 5995, 70, 3633, 1674, ...","[1913, 1419, 4927, 50068, 7700, 1757, 5300, 51..."
...,...,...,...
2059,71497,"[5989, 5418, 1291, 1089, 1214, 1682, 4848, 325...","[1608, 3175, 2006, 5418, 1909, 5989, 4246, 549..."
2060,71509,"[46578, 1270, 5064, 2019, 1961, 175, 6711, 415...","[2019, 2238, 6783, 1231, 8914, 53887, 2010, 10..."
2061,71525,"[8360, 7438, 2019, 1682, 47099, 541, 47044, 48...","[49530, 47099, 49278, 7147, 51575, 48304]"
2062,71529,"[223, 46578, 3996, 1220, 2671, 104, 913, 253, ...","[3996, 5952, 786, 1917, 2355, 1682]"


In [23]:
user_dps_df = evaluator.eval_user_diversity_preference_scale(encoded_train_df, feature_engineer.vocab2idx, normalized=True)
user_dps_df.head(1)

Calculating user diversity preference scale:   0%|          | 0/2064 [00:00<?, ?it/s]

Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:06<00:00, 315.91it/s]


,userID,actorID_wvec,actorID_dps,country_wvec,country_dps,directorID_wvec,directorID_dps,genre_wvec,genre_dps
0,0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, ...",0.577311,"[0.0, 0.0, 0.0, 0.0, 0.0, 4.0, 0.0, 0.0, 0.0, ...",0.183214,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.494533,"[97.0, 41.5, 8.0, 5.0, 39.0, 45.5, 0.0, 44.5, ...",0.755545


In [24]:
user_dpms_df = evaluator.evaluate_dpms_at_k(
    eval_df=eval_df,
    feature_engineer=feature_engineer,
    ground_truth_dps_df=user_dps_df,
    k=20,
    actor_k=5,
    rare_threshold=5,
)

user_dpms_df.describe()

exploded 2064
extracting item features...
merging features...
interaction data count before merging: 41280
interaction data count after merging: 41280
done!
Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre
encoded 2064


Calculating user diversity preference scale: 100%|██████████| 2064/2064 [00:03<00:00, 650.71it/s]


combined_df 2064
Calculating DPMS for each feature...


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.171185,0.975208,0.179043,0.884949,0.552596
std,595.969798,0.076487,0.032692,0.104933,0.074582,0.054589
min,0.000000,0.000000,0.479621,0.000000,0.406314,0.327811
25%,515.750000,0.114853,0.971566,0.102149,0.853219,0.517791
50%,1031.500000,0.172197,0.984090,0.175461,0.905436,0.555591
75%,1547.250000,0.227582,0.991111,0.251628,0.936727,0.590855
max,2063.000000,0.421560,0.999087,0.536480,0.987745,0.707243
